# Global News & Paper Multi-Angle Perspective Analyzer

- 이름: 쿠츠커러프 버부르벡
- 학번: 20232765

다국어 관점 차이 및 프레이밍 분석 AI — 이 노트북은 프로젝트의 핵심 아키텍처와
소스 코드를 순서대로 실행하며 설명합니다. 이 프로젝트는 **100% OpenAI API**로
동작합니다 (채팅/분석용 ChatOpenAI + 임베딩용 OpenAIEmbeddings, 별도 Anthropic
의존성 없음). 실제 OpenAI API 호출이 필요한 셀은 `OPENAI_API_KEY`가 설정된
경우에만 실행되며, 키가 없는 경우 오류 없이 건너뜁니다 (`SKIP` 메시지 출력).

**이 노트북에는 실제 API 키가 포함되어 있지 않습니다.** `.env` 파일 또는
환경 변수를 통해 키를 주입하세요.

## 1. 아키텍처 개요

```text
Source A Files
    ↓
Document Loader A
    ↓
Multilingual Chunking
    ↓
OpenAI Embeddings
    ↓
FAISS Vector Store A
    ↓
Retriever A ───────────────┐
                           ├── OpenAI (ChatOpenAI) Analysis Chain
Retriever B ───────────────┘
    ↑
FAISS Vector Store B
    ↑
Multilingual Chunking
    ↑
Document Loader B
    ↑
Source B Files
```

핵심 설계 원칙:
1. Source A와 Source B는 검색 시점까지 완전히 분리된 별도의 FAISS 인덱스에 저장됩니다.
2. 모든 비교/질의응답 요청은 두 저장소에서 **독립적으로** `k=4~6`개씩 검색한 뒤 결과를 결합합니다.
3. OpenAI 채팅 모델 id는 하드코딩하지 않고 `OPENAI_MODEL` 환경변수로 관리합니다 (미설정 시 안전한 기본값 `gpt-4o-mini` 사용).

## 2. 프로젝트 폴더 구조 생성

아래 셀은 순차 실행 시 필요한 폴더/파일이 없으면 생성합니다 (이미 존재하면 건너뜁니다).
저장소를 처음부터 재구성하는 상황을 시연하기 위한 셀입니다.

In [1]:
import os

PROJECT_ROOT = os.getcwd()

REQUIRED_DIRS = ["utils", "prompts", "sample_data", "tests", ".streamlit"]
for d in REQUIRED_DIRS:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)
    print(f"OK: {path}")

for pkg in ["utils", "prompts", "tests"]:
    init_path = os.path.join(PROJECT_ROOT, pkg, "__init__.py")
    if not os.path.exists(init_path):
        with open(init_path, "w") as f:
            f.write("")
        print(f"created {init_path}")
    else:
        print(f"exists  {init_path}")

OK: /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/utils
OK: /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/prompts
OK: /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/sample_data
OK: /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/tests
OK: /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/.streamlit
exists  /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/utils/__init__.py
exists  /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/prompts/__init__.py
exists  /Users/boburbek/Desktop/MINI_PRO/global-perspective-rag/tests/__init__.py


## 3. 환경 변수 로드

`.env` 파일이 있으면 로드합니다. 이 노트북 자체에는 실제 키 값을 절대 적지 않습니다.

In [2]:
from dotenv import load_dotenv

load_dotenv()

HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))

print("OPENAI_API_KEY set:", HAS_OPENAI_KEY)
print("OPENAI_MODEL (env, optional):", os.getenv("OPENAI_MODEL", "<unset - safe default will be used>"))
print("OPENAI_EMBEDDING_MODEL (env, optional):", os.getenv("OPENAI_EMBEDDING_MODEL", "<unset - safe default will be used>"))

OPENAI_API_KEY set: False
OPENAI_MODEL (env, optional): <unset - safe default will be used>
OPENAI_EMBEDDING_MODEL (env, optional): <unset - safe default will be used>


## 4. 핵심 소스 코드

아래 셀들은 `utils/document_loader.py`, `utils/vector_store.py`, `utils/rag_chain.py`,
`prompts/*.py`에 있는 것과 동일한 핵심 로직을 그대로 임포트하여 사용합니다.
(중복 유지보수를 피하기 위해 코드를 복사하지 않고 실제 모듈을 import 합니다.)

In [3]:
from utils.document_loader import (
    EmptyDocumentError,
    UnsupportedFileTypeError,
    chunk_documents,
    load_file_to_documents,
    process_uploaded_files,
)
from utils.vector_store import (
    build_vector_store,
    compute_documents_hash,
    format_docs_for_prompt,
    retrieve_from_both,
)
from utils.rag_chain import (
    DEFAULT_OPENAI_MODEL,
    build_comparison_chain,
    build_qa_chain,
    get_model_name,
)

print("Default OpenAI chat model (used only if OPENAI_MODEL is unset):", DEFAULT_OPENAI_MODEL)
print("Model that will actually be used this run:", get_model_name())

Default OpenAI chat model (used only if OPENAI_MODEL is unset): gpt-4o-mini
Model that will actually be used this run: gpt-4o-mini


## 5. 샘플 데이터 로드 (Source A / Source B)

In [4]:
with open("sample_data/us_tech_regulation.txt", "rb") as f:
    docs_a = load_file_to_documents(
        f, filename="us_tech_regulation.txt", source_label="Source A", language="English"
    )

with open("sample_data/kr_tech_regulation.txt", "rb") as f:
    docs_b = load_file_to_documents(
        f, filename="kr_tech_regulation.txt", source_label="Source B", language="한국어"
    )

print(f"Source A pages loaded: {len(docs_a)}")
print(f"Source B pages loaded: {len(docs_b)}")
print("Sample Source A snippet:\n", docs_a[0].page_content[:200])

Source A pages loaded: 1
Source B pages loaded: 1
Sample Source A snippet:
 [SYNTHETIC DEMONSTRATION CONTENT — This article was written for educational
and testing purposes for a RAG demo project. It does not represent a real
publication, real reporters, or real news organiza


## 6. 청크 분할 테스트 (Chunking)

In [5]:
chunks_a = chunk_documents(docs_a, source_letter="A")
chunks_b = chunk_documents(docs_b, source_letter="B")

print(f"Source A chunks: {len(chunks_a)}")
print(f"Source B chunks: {len(chunks_b)}")
print()
print("Example chunk metadata (Source A, chunk 0):")
print(chunks_a[0].metadata)
assert chunks_a[0].metadata["chunk_id"].startswith("A-")
assert chunks_b[0].metadata["chunk_id"].startswith("B-")
print("\nOK: chunk_id prefixes confirm Source A / Source B are kept separate.")

Source A chunks: 9
Source B chunks: 3

Example chunk metadata (Source A, chunk 0):
{'source_label': 'Source A', 'filename': 'us_tech_regulation.txt', 'language': 'English', 'page': 1, 'chunk_id': 'A-001-01'}

OK: chunk_id prefixes confirm Source A / Source B are kept separate.


## 7. 벡터 저장소 구축 (Source A / Source B, 독립적으로)

`OPENAI_API_KEY`가 설정되어 있어야 실행됩니다 (채팅과 임베딩 모두 이 하나의
키로 동작합니다). 키가 없으면 이 셀은 안전하게 건너뛰고 `SKIP`을 출력합니다 —
이는 API 통합을 실제로 검증하지 못했음을 투명하게 보고하기 위함입니다.

In [6]:
store_a = None
store_b = None

if HAS_OPENAI_KEY:
    store_a = build_vector_store(chunks_a, source_label="Source A")
    store_b = build_vector_store(chunks_b, source_label="Source B")
    print("Vector Store A hash:", store_a.documents_hash[:12], "...")
    print("Vector Store B hash:", store_b.documents_hash[:12], "...")
    print("두 벡터 저장소가 독립적으로 생성되었습니다.")
else:
    print("SKIP: OPENAI_API_KEY가 설정되지 않아 임베딩/벡터 저장소 생성을 건너뜁니다.")
    print("      (실제 배포 전 반드시 유효한 키로 이 셀을 실행하여 검증하세요.)")

SKIP: OPENAI_API_KEY가 설정되지 않아 임베딩/벡터 저장소 생성을 건너뜁니다.
      (실제 배포 전 반드시 유효한 키로 이 셀을 실행하여 검증하세요.)


## 8. 검색된 근거의 소스 메타데이터 확인

In [7]:
if store_a is not None and store_b is not None:
    query = "AI regulation impact on startups and consumer protection"
    retrieved = retrieve_from_both(store_a, store_b, query, k=4)
    for doc in retrieved:
        print(doc.metadata)
    print("\n--- Formatted context sent to the LLM ---\n")
    print(format_docs_for_prompt(retrieved)[:800], "...")
else:
    print("SKIP: 벡터 저장소가 생성되지 않아 검색 예시를 건너뜁니다.")

SKIP: 벡터 저장소가 생성되지 않아 검색 예시를 건너뜁니다.


## 9. 비교 분석 체인 실행 예시 (Comparative Analysis Chain)

`OPENAI_API_KEY`가 설정되어 있어야 실행됩니다 (ChatOpenAI를 사용).

In [8]:
if store_a is not None and store_b is not None and HAS_OPENAI_KEY:
    comparison_chain = build_comparison_chain(store_a, store_b)
    result = comparison_chain.invoke({"topic": "AI regulation: innovation vs. consumer protection"})
    print(result["report"])
else:
    print("SKIP: OPENAI_API_KEY 또는 벡터 저장소가 준비되지 않아 비교 분석 실행을 건너뜁니다.")
    print("      (실제 OpenAI API 통합은 유효한 키가 있는 환경에서 별도로 검증해야 합니다.)")

SKIP: OPENAI_API_KEY 또는 벡터 저장소가 준비되지 않아 비교 분석 실행을 건너뜁니다.
      (실제 OpenAI API 통합은 유효한 키가 있는 환경에서 별도로 검증해야 합니다.)


## 10. 다국어 Q&A 체인 실행 예시

In [9]:
if store_a is not None and store_b is not None and HAS_OPENAI_KEY:
    qa_chain = build_qa_chain(store_a, store_b)
    qa_result = qa_chain.invoke(
        {
            "question": "두 문서는 스타트업에 미치는 영향을 어떻게 다르게 설명하나요?",
            "output_language": "한국어",
        }
    )
    print(qa_result["answer"])
else:
    print("SKIP: OPENAI_API_KEY 또는 벡터 저장소가 준비되지 않아 Q&A 실행을 건너뜁니다.")

SKIP: OPENAI_API_KEY 또는 벡터 저장소가 준비되지 않아 Q&A 실행을 건너뜁니다.


## 11. 테스트 실행 (터미널에서)

```bash
python -m compileall .
python -m pytest tests -q
```

두 명령 모두 API 키 없이 실행 가능하며, 문서 로딩/청크/메타데이터/모듈 임포트를
검증합니다. 실제 검증 결과는 `PROJECT_REPORT.docx` / `PROJECT_REPORT_DRAFT.md`의
'Testing Results' 섹션을 참고하세요.

## 12. 배포 안내 (Streamlit Community Cloud)

1. GitHub 저장소 생성 후 코드 푸시:
   ```bash
   git init
   git add .
   git commit -m "Initial commit"
   git branch -M main
   git remote add origin <YOUR_GITHUB_REPO_URL>
   git push -u origin main
   ```
2. https://share.streamlit.io 에서 New app 생성
3. Repository / Branch 선택, **Main file path: `app.py`**
4. App settings → Secrets에 다음을 등록:
   ```toml
   OPENAI_API_KEY = "sk-..."
   OPENAI_MODEL = "gpt-4o-mini"
   ```
5. Deploy 후 공개 URL이 정상 동작하는지 확인 (일반 사용자는 API 키 입력 불필요)

## 13. 최종 제출 체크리스트

```text
[ ] Name and student ID inserted
[ ] Application runs locally
[ ] Source A and Source B retrieve independently
[ ] Evidence references appear in answers
[ ] GitHub repository pushed
[ ] Streamlit public URL works
[ ] project_code.ipynb completed
[ ] PROJECT_REPORT.docx completed
[ ] No API keys committed
[ ] Slack submission message prepared
```